# 09 — Image Similarity Search with LanceDB (Student Version)

In this notebook, you'll build your first Image Similarity Search system. 
We'll focus on the **3 main AI steps**: loading the model, generating vectors, and searching.

Everything else (imports, database setup) is pre-filled for you!

## 1) Setup & Loading Data

First, we'll import our tools and load any animal images we've placed in the `images/` folder.

In [ ]:
import os
from pathlib import Path
import lancedb
from PIL import Image
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

# 1. Setup paths
BASE_DIR = Path.cwd()
IMAGE_DIR = BASE_DIR / "images"
DB_PATH = str(BASE_DIR / "lancedb_images")

# 2. Find and preview images
image_paths = list(IMAGE_DIR.glob("*.png")) + list(IMAGE_DIR.glob("*.jpg"))
print(f"Found {len(image_paths)} images.")

fig, axes = plt.subplots(1, len(image_paths), figsize=(15, 5))
for i, p in enumerate(image_paths):
    axes[i].imshow(Image.open(p))
    axes[i].axis('off')
plt.show()

## 🛠️ Step 1: Load the CLIP Model

**Your Task**: Load the `clip-ViT-B-32` model using `SentenceTransformer`.

In [ ]:
# TODO: Load the CLIP ViT-B-32 model here
model = None # Change this line!

print("Model loaded successfully!")

## 🛠️ Step 2: Generate Vectors (Embeddings)

**Your Task**: Turn all your images into a list of vectors using your model.

In [ ]:
print("Generating embeddings...")
images = [Image.open(p) for p in image_paths]

# TODO: Use your model to encode all the images into vectors
vectors = [] # Update this line!

print(f"Generated {len(vectors)} vectors with {len(vectors[0]) if len(vectors) > 0 else 'N/A'} dimensions each.")

## 3) Storage (LanceDB)

We'll now pair these vectors with their filenames and store them in **LanceDB**. 
This part is pre-filled as it is "bookkeeping"!

In [ ]:
# Pair images and vectors
data = [
    {"path": str(p), "vector": v.tolist()}
    for p, v in zip(image_paths, vectors)
]

# Connect to LanceDB and create a search table
if os.path.exists(DB_PATH):
    import shutil
    shutil.rmtree(DB_PATH)

db = lancedb.connect(DB_PATH)
table = db.create_table("images", data=data)

print("Knowledge base ready for search!")

## 🛠️ Step 3: Semantic Search

**Your Task**: Search for images using a simple text query.

In [ ]:
def search_by_text(query, limit=1):
    # Embed the text query
    query_vector = model.encode([query])[0]
    # Ask the database for the closest match
    results = table.search(query_vector).limit(limit).to_pandas()
    
    # Display the result
    for _, row in results.iterrows():
        img = Image.open(row['path'])
        plt.imshow(img)
        plt.title(f"Query: '{query}'\nMatch: {Path(row['path']).name}")
        plt.axis('off')
        plt.show()

# TODO: Try searching for "a photo of a cat" or another animal you have!
search_by_text("???")

### 🎉 You did it!
You've built an AI-powered image search system that understands the **content** of photos, not just their filenames!